In [1]:
import os
import json
import numpy as np
import pandas as pd
import pyvista as pv
import pvlib
from shapely.geometry import Polygon, Point
from earcut import earcut
import time
import matplotlib.pyplot as plt

## 1. Configurations and File Paths

We set the parameters for our target building and location:
- **Site location**: Würzburg, Germany (Latitude: 49.77243, Longitude: 9.87073, Timezone: 'Europe/Berlin').
- **Target Building ID**: `DEBY_LOD2_506720`.
- **lookng for shadow/raytracing in**: 100 meters.
- **Time steps**: 10-minute intervals for the full year.

In [2]:
LATITUDE = 49.77243
LONGITUDE = 9.87073
TIMEZONE = 'Europe/Berlin'
TARGET_BUILDING_ID = 'DEBY_LOD2_506720'
OBSTACLE_RADIUS_M = 100.0

if os.path.basename(os.getcwd()) == "notebooks":
    DATA_DIR = "../outputs"
else:
    DATA_DIR = "outputs"

TREES_CSV = os.path.join(DATA_DIR, "wuerzburg_extracted_trees.csv")
ROOFS_CSV = os.path.join(DATA_DIR, "wuerzburg_roof_details.csv")
SURFACES_JSON = os.path.join(DATA_DIR, "cityGML_obj_surfaces.json")
OBJECTS_CSV = os.path.join(DATA_DIR, "cityGML_objects.csv")

print(f"Loading data from: {DATA_DIR}")

Loading data from: ../outputs


## 2. Helper Functions for Geometry and Point Sampling

We define simple helper functions to load spatial data, fit a plane to the roof vertices for Z-interpolation, and sample a grid of points on the roof surface.

In [3]:
def parse_coords(s: str) -> np.ndarray:
    """Parses a space-separated string of coords into an (N, 3) numpy array."""
    vals = list(map(float, s.split()))
    pts = np.array(vals).reshape(-1, 3)
    if len(pts) > 1 and np.allclose(pts[0], pts[-1]):
        pts = pts[:-1]
    return pts

def sample_grid_points_on_roof(roof_coords_str: str) -> np.ndarray:
    """Samples points on the roof: centroid and a regular grid inside the polygon."""
    v = parse_coords(roof_coords_str)
    
    sampled = [v.mean(axis=0)]
    
    min_x, max_x = v[:, 0].min(), v[:, 0].max()
    min_y, max_y = v[:, 1].min(), v[:, 1].max()
    
    A = np.column_stack([v[:, 0], v[:, 1], np.ones(len(v))])
    b = v[:, 2]
    coeff, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    
    poly_2d = Polygon(v[:, :2])
    
    xs = np.linspace(min_x, max_x, 7)[1:-1]
    ys = np.linspace(min_y, max_y, 7)[1:-1]
    
    for x_g in xs:
        for y_g in ys:
            if poly_2d.contains(Point(x_g, y_g)):
                z_g = coeff[0]*x_g + coeff[1]*y_g + coeff[2]
                sampled.append(np.array([x_g, y_g, z_g]))
                
    unique_pts = []
    for p in sampled:
        if not any(np.allclose(p, up) for up in unique_pts):
            unique_pts.append(p)
            
    return np.array(unique_pts)

## 3. Data Loading and Spatial Filtering

We load the building data and tree details. Using the centroid of our target building, we filter obstacles within our search radius (100 meters). The target building itself is excluded from the obstacle lists.

In [4]:
df_objects = pd.read_csv(OBJECTS_CSV)
df_trees = pd.read_csv(TREES_CSV)
df_roofs = pd.read_csv(ROOFS_CSV)

# Target building details
target_row = df_objects[df_objects['CityGML_ID'] == TARGET_BUILDING_ID].iloc[0]
cx, cy = target_row['Centroid x UTM'], target_row['Centroid y UTM']
lat = target_row['Centroid Latitude']
lon = target_row['Centroid Longitude']
print(f"Target building centroid: ({cx:.2f}, {cy:.2f})")

# Filter surrounding buildings within 100m, excluding the target building itself
dists_b = np.sqrt((df_objects['Centroid x UTM'] - cx)**2 + (df_objects['Centroid y UTM'] - cy)**2)
nearby_b_ids = df_objects[
    (dists_b <= OBSTACLE_RADIUS_M) & 
    (df_objects['CityGML_ID'] != TARGET_BUILDING_ID)
]['CityGML_ID'].tolist()
print(f"Found {len(nearby_b_ids)} surrounding buildings within {OBSTACLE_RADIUS_M}m.")

# Filter trees within 100m
dists_t = np.sqrt((df_trees['Tree_Position_X'] - cx)**2 + (df_trees['Tree_Position_Y'] - cy)**2)
df_nearby_trees = df_trees[dists_t <= OBSTACLE_RADIUS_M]
print(f"Found {len(df_nearby_trees)} trees within {OBSTACLE_RADIUS_M}m.")

Target building centroid: (562696.35, 5513692.88)
Found 48 surrounding buildings within 100.0m.
Found 51 trees within 100.0m.


## 4. Constructing the Simplified 3D Obstacle Scene

We build a combined obstacle mesh in PyVista representing the surrounding environment (excluding the target building itself).

In [5]:
with open(SURFACES_JSON, 'r') as f:
    surfaces_data = json.load(f)

# Taking surrounding buildings as simple 3D boxes
building_meshes = []
for b_id in nearby_b_ids:
    if b_id in surfaces_data:
        all_pts = []
        b_data = surfaces_data[b_id]
        for s_type in ['CoordsGroundSurfaces', 'CoordsWallSurfaces', 'CoordsRoofSurfaces']:
            if s_type in b_data:
                for s_str in b_data[s_type]:
                    all_pts.extend(parse_coords(s_str))
        if len(all_pts) == 0:
            continue
        all_pts = np.array(all_pts)
        
        # Build axis-aligned box representing the building
        min_x, max_x = all_pts[:, 0].min(), all_pts[:, 0].max()
        min_y, max_y = all_pts[:, 1].min(), all_pts[:, 1].max()
        min_z, max_z = all_pts[:, 2].min(), all_pts[:, 2].max()
        if max_z == min_z:
            max_z += 5.0  # fallback height
            
        box_mesh = pv.Box(bounds=(min_x, max_x, min_y, max_y, min_z, max_z))
        building_meshes.append(box_mesh)
print(f"Built {len(building_meshes)} building boxes.")

# trees as vertical cones
tree_meshes = []
for idx, row in df_nearby_trees.iterrows():
    h = row['cone_height']
    r = row['cone_radius']
    tx = row['Tree_Position_X']
    ty = row['Tree_Position_Y']
    tz_ground = row['dgmhoehe']
    center = (tx, ty, tz_ground + h / 2.0)
    
    cone = pv.Cone(center=center, direction=(0, 0, 1), height=h, radius=r, resolution=16)
    tree_meshes.append(cone)
print(f"Built {len(tree_meshes)} tree cones.")

# Merge obstacles into a single obstacle mesh for ray-tracing
obstacle_mesh = pv.merge(building_meshes + tree_meshes)

Built 48 building boxes.
Built 51 tree cones.


## 5. Annual Solar Vector Calculations

We calculate the solar angles using `pvlib` and convert them to 3D unit direction vectors.

In [6]:
# Generate timeline for the full year at 10-minute intervals
timeline = pd.date_range(start='2026-01-01 00:00:00', end='2026-12-31 23:50:00', freq='10min', tz=TIMEZONE)
print(f"Timeline steps: {len(timeline)}")

# solar position
solar_pos = pvlib.solarposition.get_solarposition(timeline, lat, lon)

df_solar = pd.DataFrame(index=timeline)
df_solar['elevation'] = solar_pos['elevation']
df_solar['azimuth'] = solar_pos['azimuth']
df_solar['is_day'] = df_solar['elevation'] > 0.0

# Convert angles to 3D direction vectors pointing to the Sun
el_rad = np.radians(df_solar['elevation'])
az_rad = np.radians(df_solar['azimuth'])

df_solar['sun_dx'] = np.cos(el_rad) * np.sin(az_rad)
df_solar['sun_dy'] = np.cos(el_rad) * np.cos(az_rad)
df_solar['sun_dz'] = np.sin(el_rad)

sun_vectors = df_solar[['sun_dx', 'sun_dy', 'sun_dz']].values
daytime_indices = np.where(df_solar['is_day'].values)[0]
print(f"Daytime steps: {len(daytime_indices)} ({len(daytime_indices)/len(timeline)*100:.1f}%)")

Timeline steps: 52560
Daytime steps: 26426 (50.3%)


## 6. Sampling Points on Target Roof

We sample points (the centroid and a small grid) across the target building's roof facets from `wuerzburg_roof_details.csv`.

In [7]:
target_roofs = df_roofs[df_roofs['Building_ID'] == TARGET_BUILDING_ID]
print(f"Target building {TARGET_BUILDING_ID} has {len(target_roofs)} roof surfaces.")

roof_points = []
roof_labels = []

for _, row in target_roofs.iterrows():
    r_idx = row['Roof_Index']
    coords_str = row['roof_coords']
    pts = sample_grid_points_on_roof(coords_str)
    
    print(f"Roof Index {r_idx}: Sampled {len(pts)} points.")
    for i, pt in enumerate(pts):
        roof_points.append(pt)
        roof_labels.append(f"Roof_{r_idx}_Pt_{i}")

roof_points = np.array(roof_points)
print(f"Total roof sample points: {len(roof_points)}")

Target building DEBY_LOD2_506720 has 2 roof surfaces.
Roof Index 0: Sampled 14 points.
Roof Index 1: Sampled 14 points.
Total roof sample points: 28


## 7. Ray-Tracing Shading Simulation Loop

We perform ray-tracing for each roof point during daytime hours. If a ray intersects the obstacle mesh, the point is shaded.

In [8]:
# Initialize shading matrix (True means shaded, False means sunlit)
shading_matrix = np.ones((len(timeline), len(roof_points)), dtype=bool)

start_time = time.time()

for step_idx in daytime_indices:
    sun_vec = sun_vectors[step_idx]
    
    for pt_idx, pt in enumerate(roof_points):
        # Nudge origin slightly along the sun vector to avoid self-intersection
        origin = pt + sun_vec * 0.05
        end_point = origin + sun_vec * 1000.0
        
        pts, cells = obstacle_mesh.ray_trace(origin, end_point)
        if len(pts) == 0:
            shading_matrix[step_idx, pt_idx] = False
            
print(f"Simulation completed")

Simulation completed


## 8. Summary of Results for the Advisor

In this section, we analyze the shading data and produce useful outputs for the advisor:
1. **Summary Table**: Replaces overall metrics with daylight-only metrics and ranks the points based on sunlit percentage.
2. **Actionable Recommendations**: Displays the best point and lists the top 3 points.
3. **Roof-wise Summary**: Computes roof-facet level averages to evaluate which roof facet is overall better.

In [9]:
# Compute summary statistics
summary_rows = []
daytime_total = len(daytime_indices)

for pt_idx, pt in enumerate(roof_points):
    daytime_shaded = shading_matrix[daytime_indices, pt_idx].sum()
    daytime_sunlit = daytime_total - daytime_shaded
    
    # Daylight hours calculation (10-minute intervals)
    daylight_sunlit_hours = daytime_sunlit * 10 / 60.0
    daylight_shaded_hours = daytime_shaded * 10 / 60.0
    sunlit_percent_daylight = (daytime_sunlit / daytime_total) * 100.0
    
    summary_rows.append({
        'point_id': roof_labels[pt_idx],
        'x': pt[0],
        'y': pt[1],
        'daylight_sunlit_hours': round(daylight_sunlit_hours, 1),
        'daylight_shaded_hours': round(daylight_shaded_hours, 1),
        'sunlit_percent_daylight': round(sunlit_percent_daylight, 2)
    })

# Create summary DataFrame and add ranking
df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values(
    'sunlit_percent_daylight',
    ascending=False
).reset_index(drop=True)

df_summary['rank'] = df_summary.index + 1

# Display summary table
print("Ranked Roof Points Shading Summary (Daylight Only):")
display(df_summary)

# ROOF RECOMMENDATIONS
best_point = df_summary.loc[0]

print("\n=== Solar Panel Recommendation ===")
print(f"Best location: {best_point['point_id']}")
print(f"Daylight sun exposure: {best_point['sunlit_percent_daylight']:.1f}%")
print(f"Annual daylight sunlit hours: {best_point['daylight_sunlit_hours']:.1f} h")

top_3 = df_summary.head(3)
print("\nTop 3 recommended points:")
display(top_3[['point_id', 'sunlit_percent_daylight', 'daylight_sunlit_hours']])

# ROOF-WISE SUMMARY 
if len(target_roofs) > 1:
    df_summary['roof_id'] = df_summary['point_id'].str.extract(r'(Roof_\d+)')
    
    roof_summary = df_summary.groupby('roof_id').agg({
        'sunlit_percent_daylight': 'mean',
        'daylight_sunlit_hours': 'mean',
        'point_id': 'count'
    }).rename(columns={'point_id': 'num_points'})

    roof_summary = roof_summary.sort_values(
        'sunlit_percent_daylight',
        ascending=False
    )
    
    print("\nRoof-wise average sun exposure:")
    display(roof_summary)

Ranked Roof Points Shading Summary (Daylight Only):


,point_id,x,y,daylight_sunlit_hours,daylight_shaded_hours,sunlit_percent_daylight,rank
0,Roof_0_Pt_9,562696.468000,5.513695e+06,4372.8,31.5,99.28,1
1,Roof_0_Pt_12,562698.820500,5.513697e+06,4371.5,32.8,99.25,2
2,Roof_1_Pt_2,562693.805667,5.513688e+06,4370.0,34.3,99.22,3
3,Roof_0_Pt_6,562694.115500,5.513692e+06,4370.2,34.2,99.22,4
4,Roof_1_Pt_8,562698.511000,5.513693e+06,4368.2,36.2,99.18,5
5,Roof_0_Pt_3,562691.763000,5.513690e+06,4367.8,36.5,99.17,6
6,Roof_1_Pt_5,562696.158333,5.513691e+06,4367.2,37.2,99.16,7
7,Roof_1_Pt_11,562700.863667,5.513696e+06,4364.8,39.5,99.10,8
8,Roof_0_Pt_13,562698.820500,5.513700e+06,4349.8,54.5,98.76,9
9,Roof_0_Pt_10,562696.468000,5.513697e+06,4347.7,56.7,98.71,10



=== Solar Panel Recommendation ===
Best location: Roof_0_Pt_9
Daylight sun exposure: 99.3%
Annual daylight sunlit hours: 4372.8 h

Top 3 recommended points:


,point_id,sunlit_percent_daylight,daylight_sunlit_hours
0,Roof_0_Pt_9,99.28,4372.8
1,Roof_0_Pt_12,99.25,4371.5
2,Roof_1_Pt_2,99.22,4370.0



Roof-wise average sun exposure:


,sunlit_percent_daylight,daylight_sunlit_hours,num_points
roof_id,,,
Roof_0,98.507857,4338.671429,14
Roof_1,98.120714,4321.564286,14
